# SRM Insider - Bike Rental Analysis

This notebook performs an analysis of bike rental patterns and builds a predictive model using the Bike Sharing Dataset from the UCI Machine Learning Repository.

## 1. Setup and Dependencies

First, we install necessary libraries and import our core tools for data manipulation, visualization, and machine learning.

In [ ]:
# Install requirements
!pip install statsmodels

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import seasonal_decompose
from pandas.plotting import autocorrelation_plot

## 2. Data Acquisition

We download the dataset directly from the UCI repository and extract the contents.

In [ ]:
# Fetching the dataset
!wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip
!unzip -o -q Bike-Sharing-Dataset.zip

## 3. Data Preprocessing

We load the hourly data, convert time information into a proper datetime format, and remove unneeded columns to prepare for analysis.

In [ ]:
# Load and clean data
import pandas as pd
df = pd.read_csv('hour.csv')

# Preprocess datetime
df['datetime'] = pd.to_datetime(df['dteday']) + pd.to_timedelta(df['hr'], unit='h')
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

# Drop identification and redundant columns
df = df.drop(['instant', 'dteday', 'casual', 'registered'], axis=1)

print("Data loaded and preprocessed.")
df.head()

## 4. Visualizing Rental Trends

Let's visualize the overall trend of bike rentals over time to identify any seasonality or growth.

In [ ]:
# Overall trend visualization
import matplotlib.pyplot as plt
plt.figure(figsize=(15,5))
plt.plot(df['cnt'], linewidth=0.5)
plt.title("Bike Rentals Over Time (Overall Trend)")
plt.xlabel("Time")
plt.ylabel("Number of Rentals")
plt.show()

## 5. Temporal Patterns

We analyze the average number of rentals by hour of the day to understand daily peak usage.

In [ ]:
# Analysis by hour
hourly_avg = df.groupby('hr')['cnt'].mean()
plt.figure(figsize=(10,5))
plt.plot(hourly_avg, marker='o')
plt.title("Average Bike Rentals by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Average Rentals")
plt.grid(True)
plt.show()

## 6. Model Training and Prediction

We split the data into training and testing sets and train a Random Forest model to predict the rental counts based on weather and temporal features.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Prepare features and target
X = df.drop(['cnt'], axis=1)
y = df['cnt']

# Data split
train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# Initialize and train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model training complete.")

## 7. Model Evaluation

Finally, we evaluate the model performance by visualizing the predicted counts against the actual rentals in the test set.

In [ ]:
from sklearn.metrics import mean_squared_error

# Prediction and comparison
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")

# Visualization of results
plt.figure(figsize=(15,5))
plt.plot(y_test.values[:100], label='Actual', marker='o')
plt.plot(y_pred[:100], label='Predicted', marker='x')
plt.title("Actual vs Predicted Bike Rentals (First 100 entries)")
plt.legend()
plt.show()